# 🚢 Titanic Dataset — Data Cleaning & Preprocessing
**Task 1 | Data Science Internship Project**

---
**Objective:** Clean and prepare the raw Titanic dataset for downstream analysis and modeling.

**Steps covered:**
1. Load & inspect raw data
2. Remove duplicates
3. Rename columns (snake_case)
4. Convert data types
5. Handle missing values
6. Feature engineering (bonus)
7. Save clean dataset + visualise cleaning summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Load & Inspect Raw Data

In [ ]:
df = pd.read_csv('titanic_raw.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicates ===')
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
print('=== Descriptive Statistics ===')
df.describe(include='all')

## 2. Remove Duplicates

In [ ]:
n_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
n_removed = n_before - len(df)
print(f'Rows before: {n_before}')
print(f'Duplicates removed: {n_removed}')
print(f'Rows after:  {len(df)}')

## 3. Rename Columns (snake_case)

In [ ]:
df.rename(columns={
    'PassengerId': 'passenger_id',
    'Survived':    'survived',
    'Pclass':      'pclass',
    'Name':        'name',
    'Sex':         'sex',
    'Age':         'age',
    'SibSp':       'siblings_spouses',
    'Parch':       'parents_children',
    'Ticket':      'ticket',
    'Fare':        'fare',
    'Cabin':       'cabin',
    'Embarked':    'embarked'
}, inplace=True)
print('Renamed columns:', df.columns.tolist())

## 4. Convert Data Types

In [ ]:
# Ensure numeric columns are correct type
df['age']  = pd.to_numeric(df['age'],  errors='coerce')
df['fare'] = pd.to_numeric(df['fare'], errors='coerce')

# Ordinal/binary columns
df['survived'] = df['survived'].astype(int)
df['pclass']   = df['pclass'].astype(int)

# Categorical columns
df['sex']      = df['sex'].astype('category')
df['embarked'] = df['embarked'].astype('category')

print('Updated dtypes:')
print(df.dtypes)

## 5. Handle Missing Values

In [ ]:
missing_before = df.isnull().sum()
print('Missing values BEFORE cleaning:')
print(missing_before[missing_before > 0])

In [ ]:
# --- Age: median per pclass + sex group (captures class/gender age differences) ---
df['age'] = df.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# --- Fare: median per pclass (1 missing fare → use class median) ---
df['fare'] = df.groupby('pclass')['fare'].transform(
    lambda x: x.fillna(x.median())
)

# --- Embarked: fill with mode (most common port) ---
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# --- Cabin: too many missing; mark as 'Unknown' ---
df['cabin'] = df['cabin'].fillna('Unknown')

print('Missing values AFTER cleaning:')
print(df.isnull().sum().to_string())
print('\n✅ No missing values remain!')

## 6. Feature Engineering (Bonus)

In [ ]:
# Family size
df['family_size'] = df['siblings_spouses'] + df['parents_children'] + 1

# Travelling alone flag
df['is_alone'] = (df['family_size'] == 1).astype(int)

# Age group buckets
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'YoungAdult', 'MiddleAged', 'Senior']
)

# Fare quartile band
df['fare_band'] = pd.qcut(df['fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

# Cabin known flag (proxy for wealth/booking status)
df['cabin_known'] = (df['cabin'] != 'Unknown').astype(int)

print('Final shape:', df.shape)
print('New features:', ['family_size', 'is_alone', 'age_group', 'fare_band', 'cabin_known'])
df.head()

## 7. Save Clean Dataset

In [ ]:
df.to_csv('titanic_clean.csv', index=False)
print('✅ Clean dataset saved to titanic_clean.csv')
print('Shape:', df.shape)
df.info()

## 8. Cleaning Summary Dashboard

In [ ]:
# Run titanic_cleaning.py to generate the dashboard image
# Or view the PNG: titanic_cleaning_dashboard.png
from IPython.display import Image
Image('titanic_cleaning_dashboard.png')

---
## ✅ Summary of Cleaning Steps

| Step | Action | Detail |
|------|--------|--------|
| 1 | Remove duplicates | 3 duplicate rows dropped |
| 2 | Rename columns | PascalCase → snake_case |
| 3 | Convert types | age/fare → float64, sex/embarked → category |
| 4 | Fill Age | Median by pclass + sex group |
| 5 | Fill Fare | Median by pclass |
| 6 | Fill Embarked | Mode imputation |
| 7 | Fill Cabin | 'Unknown' placeholder |
| 8 | Feature engineering | family_size, is_alone, age_group, fare_band, cabin_known |

**Clean dataset is ready for EDA and modeling.**